# Mô hình chính: 2 nhánh 2D (MaxViT-Tiny) + 1 nhánh 3D (ResNeXt3D) + CrossGate

Pipeline đầy đủ cho bài toán phân loại glaucoma từ OCT 3D:
- **Tiền xử lý:** phát hiện trục depth, chuẩn hoá trục, chiếu 2 view en-face (`slab_mip`, `aip_full`), chuẩn hoá `/255`.
- **Khử nhiễu:** huấn luyện trên **2 tập** — (1) raw nhiễu, (2) đã khử nhiễu bằng phương pháp tốt nhất **không phải BM3D**
  (mặc định **Bilateral** (`skimage`, giữ biên tốt β≈0.80)).
- **Augmentation:** lật/rot90 nhất quán giữa 3D và 2D + jitter cường độ.
- **Mô hình:** 1 nhánh 3D **ResNeXt3D @200³** + 2 nhánh 2D **MaxViT-Tiny** (`slab_mip`, `aip_full`), hợp nhất **CrossGate**.
- **Chỉ số:** acc, balanced acc, precision, recall/sensitivity, specificity, F1, MCC, AUC-ROC, PR-AUC, ECE + bootstrap CI.
- **Hậu xử lý:** chọn threshold theo Youden-J trên validation, temperature scaling calibration.
- **X-AI:** Grad-CAM 3D, Grad-CAM 2D từng view, occlusion sensitivity, integrated gradients, attention CrossGate, branch-drop.
- Lưu **tất cả** figure/metric/model lên **Drive** + **wandb**.

**Smoke test:** đặt `FINAL_SMOKE=1` (dữ liệu tổng hợp nhỏ, 1 epoch, chạy được cả CPU) để kiểm tra toàn bộ pipeline trước khi train thật.

## 1. Setup

In [ ]:
!git clone --depth 1 https://github.com/Tqhuyen/glaucoma-thesis.git /content/glaucoma-thesis 2>/dev/null || git -C /content/glaucoma-thesis pull --ff-only 2>/dev/null || true
%cd /content/glaucoma-thesis
!pip install -q timm wandb scikit-image scipy matplotlib huggingface_hub hf_transfer pandas
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os, sys, json, time, math, csv, shutil
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, "scripts")
import final_model as fm
import resolution_study as rs
try:
    import denoise_torch as dt
except Exception as e:
    dt = None
    print("[warn] denoise_torch unavailable:", e)
try:
    import compare_denoise_methods as cdm
except Exception as e:
    cdm = None
    print("[warn] compare_denoise_methods unavailable:", e)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEVICE)
if DEVICE.type != "cuda":
    print("[warn] no GPU: real training sẽ rất chậm; dùng FINAL_SMOKE=1 để kiểm tra")


## 2. Cấu hình

In [ ]:
SMOKE = os.environ.get("FINAL_SMOKE", "0") == "1"
DATASETS = ["raw", "bilateral"]
N_2D = 2
D_LATENT = 256
ENC2D = "maxvit_tiny_rw_224"
ENC2D_PRETRAINED = not SMOKE
STORE_RES = 96 if SMOKE else 200
RES3D = 96 if SMOKE else 200
RES2D = 224
EPOCHS = 1 if SMOKE else 30
BS = 2
GRAD_ACCUM = 8
LR = 2e-4
WD = 1e-4
PATIENCE = 6
SEEDS = [42] if SMOKE else [42, 43, 44]
DENOISE_METHOD = "bilateral"
WORKERS = max(2, (os.cpu_count() or 2))
HF_RAW_REPO = "tqhuyen/harvard-oct-glaucoma-200"
HF_DN_REPO = ""
DATA_ROOT = "/tmp/final_smoke" if SMOKE else "/content/final_data"

FIG_DIR = os.path.join("figures", "final_model")
os.makedirs(FIG_DIR, exist_ok=True)
DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/MasterBKDN/Thesis")
DRIVE_DIR = os.path.join(DRIVE_ROOT, "final_2x2d_3d_crossgate")
DRIVE_FIG = os.path.join(DRIVE_ROOT, "final_2x2d_3d_crossgate_figures")
SPLITS = ("Training", "Validation", "Test")
print("smoke=", SMOKE, "| datasets=", DATASETS, "| n_2d=", N_2D, "| res3d=", RES3D, "| seeds=", SEEDS)


## 3. Dữ liệu: raw + denoised (DnCNN) + cache view

- Raw: `snapshot_download(HF_RAW_REPO)` → `{split}_volumes.npy`.
- Denoised: nếu `HF_DN_REPO` có sẵn thì tải; nếu không, **build bằng Bilateral** (`denoise_torch`, resume-safe).
- View cache: `{split}_views.npy` (2 view @224) + `{split}_dzs.npy` (trục depth) — build 1 lần.

In [ ]:
from huggingface_hub import login, snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)


def _find(data_root, split):
    return os.path.join(data_root, f"{split}_volumes.npy")


def prepare_raw():
    if SMOKE:
        return
    if all(os.path.isfile(_find(DATA_ROOT, s)) for s in SPLITS):
        print("[data] raw already present")
        return
    os.makedirs(DATA_ROOT, exist_ok=True)
    snapshot_download(repo_id=HF_RAW_REPO, repo_type="dataset", local_dir=DATA_ROOT)


def build_view_cache(data_root):
    for split in SPLITS:
        vp = os.path.join(data_root, f"{split}_views.npy")
        dzp = os.path.join(data_root, f"{split}_dzs.npy")
        if os.path.isfile(vp) and os.path.isfile(dzp):
            continue
        vols = np.load(os.path.join(data_root, f"{split}_volumes.npy"), mmap_mode="r")
        n = len(vols)
        vv = np.lib.format.open_memmap(vp, mode="w+", dtype=np.uint8, shape=(n, N_2D, RES2D, RES2D))
        dzs = np.zeros(n, dtype=np.int8)
        for i in range(n):
            raw = np.ascontiguousarray(vols[i][0])
            dz = fm.depth_axis(raw)
            dzs[i] = dz
            views = fm.project_views(fm.to_depth_last(raw, dz))
            for k in range(N_2D):
                vv[i, k] = np.clip(rs.resize_volume(views[k], (RES2D, RES2D)), 0, 255).astype(np.uint8)
            if (i + 1) % 300 == 0:
                vv.flush()
                print(f"[views] {split} {i+1}/{n}", flush=True)
        vv.flush()
        np.save(dzp, dzs)
        print(f"[views] done {split} n={n}", flush=True)


def _denoise_fn(method):
    if method in ("dncnn", "swinir"):
        if dt is None:
            raise RuntimeError("denoise_torch unavailable")
        return lambda v: dt.denoise_volume(method, v, cache_path=None)[0]
    if cdm is not None and method in cdm.METHODS:
        return lambda v: cdm.denoise_volume(v, method, workers=WORKERS, cache_path=None)[0]
    raise ValueError(f"unknown denoise method {method}")


def build_denoised(data_root, method):
    fn = _denoise_fn(method)
    for split in SPLITS:
        dp = os.path.join(data_root, f"{split}_volumes_dn.npy")
        if os.path.isfile(dp):
            continue
        vols = np.load(os.path.join(data_root, f"{split}_volumes.npy"), mmap_mode="r")
        out = np.lib.format.open_memmap(dp, mode="w+", dtype=np.uint8, shape=vols.shape)
        prog = os.path.join(data_root, f"{split}_dn_progress.json")
        done = set(json.load(open(prog)).get("i", [])) if os.path.exists(prog) else set()
        t0 = time.time()
        for i in range(len(vols)):
            if i in done:
                continue
            out[i, 0] = fn(np.ascontiguousarray(vols[i, 0]))
            done.add(i)
            if len(done) % 25 == 0 or i == len(vols) - 1:
                out.flush()
                json.dump({"i": sorted(done)}, open(prog, "w"))
                rate = (time.time() - t0) / max(len(done), 1)
                print(f"[denoise] {split} {len(done)}/{len(vols)} | {rate:.2f}s/vol", flush=True)
        out.flush()
        print(f"[denoise] done {split}", flush=True)


def make_smoke_data():
    os.makedirs(DATA_ROOT, exist_ok=True)
    rng = np.random.default_rng(0)
    for split, n in (("Training", 12), ("Validation", 6), ("Test", 6)):
        vols = rng.integers(0, 40, size=(n, 1, STORE_RES, STORE_RES, STORE_RES), dtype=np.uint8)
        for i in range(n):
            vols[i, 0, :, :, STORE_RES // 4:STORE_RES // 3] += 120
        labels = rng.integers(0, 2, size=n).astype(np.int64)
        np.save(os.path.join(DATA_ROOT, f"{split}_volumes.npy"), vols)
        np.save(os.path.join(DATA_ROOT, f"{split}_labels.npy"), labels)
        vv = np.stack([rng.integers(0, 255, (N_2D, RES2D, RES2D), dtype=np.uint8) for _ in range(n)])
        np.save(os.path.join(DATA_ROOT, f"{split}_views.npy"), vv)
        np.save(os.path.join(DATA_ROOT, f"{split}_dzs.npy"), np.full(n, 2, dtype=np.int8))
        if any(d != "raw" for d in DATASETS):
            np.save(os.path.join(DATA_ROOT, f"{split}_volumes_dn.npy"), vols)


if SMOKE:
    make_smoke_data()
    print("[smoke] synthetic data ready")
else:
    prepare_raw()
    build_view_cache(DATA_ROOT)
    if "dncnn" in DATASETS:
        if HF_DN_REPO:
            snapshot_download(repo_id=HF_DN_REPO, repo_type="dataset", local_dir=DATA_ROOT)
        else:
            build_denoised(DATA_ROOT, DENOISE_METHOD)
print("[data] root =", DATA_ROOT)


## 4. Dataset + DataLoader (augmentation nhất quán 3D↔2D)

In [ ]:
class FinalDS(Dataset):
    def __init__(self, data_root, split, denoised, train):
        self.split, self.train = split, bool(train)
        suf = "_volumes_dn.npy" if denoised else "_volumes.npy"
        self.vols = np.load(os.path.join(data_root, f"{split}{suf}"), mmap_mode="r")
        self.views = np.load(os.path.join(data_root, f"{split}_views.npy"), mmap_mode="r")
        self.dzs = np.load(os.path.join(data_root, f"{split}_dzs.npy"))
        self.labels = np.load(os.path.join(data_root, f"{split}_labels.npy"))
        self.rng = np.random.default_rng(1234 if train else 0)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        raw = np.ascontiguousarray(self.vols[i][0])
        d = int(self.dzs[i])
        if d != 2:
            raw = fm.to_depth_last(raw, d)
        views = np.asarray(self.views[i])
        if self.train:
            raw, views = fm.aug_pair(raw, views, self.rng)
        x = torch.from_numpy(raw.astype(np.float32) / 255.0)[None]
        if RES3D != raw.shape[0]:
            x = F.interpolate(x[None], size=(RES3D,) * 3, mode="trilinear", align_corners=False)[0]
        v = torch.from_numpy(views.astype(np.float32) / 255.0)[:, None]
        return x, v, torch.tensor(int(self.labels[i]), dtype=torch.long)


def make_loaders(data_root, denoised, bs):
    tr = FinalDS(data_root, "Training", denoised, True)
    va = FinalDS(data_root, "Validation", denoised, False)
    te = FinalDS(data_root, "Test", denoised, False)
    nw = 0 if os.name == "nt" else 2
    tl = DataLoader(tr, batch_size=bs, shuffle=True, num_workers=nw, pin_memory=True, drop_last=True)
    vl = DataLoader(va, batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)
    ttl = DataLoader(te, batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)
    return tl, vl, ttl


x, v, y = FinalDS(DATA_ROOT, "Training", "dncnn" in DATASETS, False)[0]
print("sample shapes:", tuple(x.shape), tuple(v.shape), int(y))


## 5. Huấn luyện + đánh giá (mỗi dataset, mỗi seed)

In [ ]:
def predict(model, loader):
    model.eval()
    ps, ys, ls = [], [], []
    with torch.no_grad():
        for x, v, y in loader:
            x, v = x.to(DEVICE), v.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEVICE.type == "cuda")):
                logits = model(x, v)
            ps.append(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
            ls.append(logits.float().cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(ps), np.concatenate(ys), np.concatenate(ls)


def train_one(data_root, denoised, seed, tag):
    torch.manual_seed(seed)
    np.random.seed(seed)
    tl, vl, ttl = make_loaders(data_root, denoised, BS)
    model = fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc2d_pretrained=ENC2D_PRETRAINED).to(DEVICE)
    ytr = np.load(os.path.join(data_root, "Training_labels.npy"))
    w = torch.tensor([len(ytr) / max(2 * int((ytr == 0).sum()), 1),
                      len(ytr) / max(2 * int((ytr == 1).sum()), 1)], dtype=torch.float32, device=DEVICE)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    steps = max(1, len(tl) // GRAD_ACCUM) * EPOCHS
    warm = int(steps * 0.05)
    sched = torch.optim.lr_scheduler.SequentialLR(
        opt, [torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=max(warm, 1)),
              torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(steps - warm, 1))], milestones=[max(warm, 1)])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    best_auc, best_state, bad, hist = -1.0, None, 0, []
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train()
        opt.zero_grad(set_to_none=True)
        run_loss = 0.0
        for i, (x, v, y) in enumerate(tl):
            x, v, y = x.to(DEVICE), v.to(DEVICE), y.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEVICE.type == "cuda")):
                loss = crit(model(x, v), y) / GRAD_ACCUM
            scaler.scale(loss).backward()
            run_loss += loss.item()
            if (i + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
        vp, vy, _ = predict(model, vl)
        vm = fm.full_metrics(vp, vy)
        hist.append({"epoch": ep + 1, "loss": run_loss / max(len(tl), 1), "val_auc": vm["auc_roc"],
                     "val_f1": vm["f1"], "val_bal": vm["balanced_acc"], "val_mcc": vm["mcc"]})
        print(f"[{tag}] ep {ep+1}/{EPOCHS} loss={hist[-1]['loss']:.4f} val_auc={vm['auc_roc']:.4f} "
              f"val_bal={vm['balanced_acc']:.4f} mcc={vm['mcc']:.4f}", flush=True)
        if vm["auc_roc"] > best_auc:
            best_auc, bad = vm["auc_roc"], 0
            best_state = {k: t.to("cpu").clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{tag}] early stop", flush=True)
                break
    model.load_state_dict(best_state)
    vp, vy, vlgt = predict(model, vl)
    T = fm.temperature_scale(vlgt, vy)
    thr = fm.tune_threshold(vp, vy)
    tp, ty, tlgt = predict(model, ttl)
    tp_cal = torch.softmax(torch.tensor(tlgt) / T, 1)[:, 1].numpy()
    res = {"tag": tag, "seed": seed, "threshold": thr, "temperature": T,
           "val": fm.full_metrics(vp, vy, thr), "test": fm.full_metrics(tp_cal, ty, thr),
           "test_ci": fm.bootstrap_ci(tp_cal, ty), "hist": hist,
           "minutes": round((time.time() - t0) / 60, 2)}
    return model, res, (tp_cal, ty), (vp, vy)


def wandb_init(name, config):
    if not os.environ.get("WANDB_API_KEY"):
        return None
    try:
        import wandb
        return wandb.init(project="glaucoma-thesis", name=name, config=config, reinit=True)
    except Exception as e:
        print("[wandb] init failed:", e)
        return None


## 6. Chạy huấn luyện trên 2 tập (raw + denoised)

In [ ]:
RESULTS = {}
for ds in DATASETS:
    denoised = (ds != "raw")
    for seed in SEEDS:
        tag = f"{ds}_s{seed}"
        wb = wandb_init("final_" + tag, {"dataset": ds, "n_2d": N_2D, "res3d": RES3D, "seed": seed})
        model, res, test_pt, val_pt = train_one(DATA_ROOT, denoised, seed, tag)
        RESULTS[tag] = {"res": res, "model_state": model.state_dict(),
                        "test_probs": test_pt[0].tolist(), "test_labels": test_pt[1].tolist()}
        if wb is not None:
            for h in res["hist"]:
                wb.log({f"train/{k}": v for k, v in h.items()}, step=h["epoch"])
            wb.log({f"test/{k}": v for k, v in res["test"].items() if isinstance(v, (int, float))})
            wb.summary.update({"test_auc": res["test"]["auc_roc"], "test_mcc": res["test"]["mcc"]})
            wb.finish()
print("done:", list(RESULTS.keys()))


## 7. Bảng chỉ số đầy đủ (val/test) + lưu CSV/JSON

In [ ]:
keys = ["acc", "balanced_acc", "precision", "recall", "specificity", "f1", "mcc", "auc_roc", "auc_pr", "ece"]
rows = []
for tag, r in RESULTS.items():
    res = r["res"]
    row = {"tag": tag, "dataset": res["tag"].split("_s")[0], "seed": res["seed"],
           "threshold": round(res["threshold"], 3), "temperature": round(res["temperature"], 3),
           "minutes": res["minutes"]}
    for split in ("val", "test"):
        for k in keys:
            row[f"{split}_{k}"] = round(res[split][k], 4)
    ci = res["test_ci"]
    row["test_auc_ci"] = f"[{ci['auc_roc'][0]:.3f},{ci['auc_roc'][1]:.3f}]"
    rows.append(row)
csv_path = os.path.join(FIG_DIR, "final_metrics.csv")
with open(csv_path, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
json.dump([r["res"] for r in RESULTS.values()], open(os.path.join(FIG_DIR, "final_metrics.json"), "w"), indent=2)
print("wrote", csv_path)
for row in rows:
    print(row["tag"], "| test auc", row["test_auc_roc"], row["test_auc_ci"],
          "| bal", row["test_balanced_acc"], "| mcc", row["test_mcc"], "| ece", row["test_ece"])


## 8. Biểu đồ: learning curves, ROC, PR, calibration, confusion

In [ ]:
def plot_report(res, probs, labels, out_prefix):
    hist = res["hist"]
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    axs[0].plot([h["epoch"] for h in hist], [h["loss"] for h in hist]); axs[0].set_title("train loss")
    axs[1].plot([h["epoch"] for h in hist], [h["val_auc"] for h in hist]); axs[1].set_title("val AUC")
    axs[2].plot([h["epoch"] for h in hist], [h["val_f1"] for h in hist]); axs[2].set_title("val F1")
    for a in axs:
        a.set_xlabel("epoch"); a.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(f"{out_prefix}_curves.png", dpi=150); plt.close(fig)

    fpr, tpr, _ = _roc(probs, labels)
    prec, rec, _ = _pr(probs, labels)
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    axs[0].plot(fpr, tpr, label=f"AUC={res['test']['auc_roc']:.3f}"); axs[0].plot([0, 1], [0, 1], "--", c="gray")
    axs[0].set_title("ROC"); axs[0].set_xlabel("FPR"); axs[0].set_ylabel("TPR"); axs[0].legend()
    axs[1].plot(rec, prec, label=f"AP={res['test']['auc_pr']:.3f}"); axs[1].set_title("Precision-Recall")
    axs[1].set_xlabel("Recall"); axs[1].set_ylabel("Precision"); axs[1].legend()
    bins = np.linspace(0, 1, 11)
    conf = np.digitize(probs, bins) - 1
    xs, ys = [], []
    for b in range(10):
        m = conf == b
        if m.sum():
            xs.append(probs[m].mean()); ys.append((labels[m] == 1).mean())
    axs[2].plot([0, 1], [0, 1], "--", c="gray"); axs[2].plot(xs, ys, "o-")
    axs[2].set_title(f"Calibration (ECE={res['test']['ece']:.3f})"); axs[2].set_xlabel("confidence")
    axs[2].set_ylabel("accuracy")
    fig.tight_layout(); fig.savefig(f"{out_prefix}_roc_pr_cal.png", dpi=150); plt.close(fig)

    pred = (probs >= res["threshold"]).astype(int)
    cm = np.array([[((pred == 0) & (labels == 0)).sum(), ((pred == 1) & (labels == 0)).sum()],
                   [((pred == 0) & (labels == 1)).sum(), ((pred == 1) & (labels == 1)).sum()]])
    fig, ax = plt.subplots(figsize=(4.2, 4))
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xlabel("pred"); ax.set_ylabel("true")
    ax.set_title("confusion (test)")
    fig.tight_layout(); fig.savefig(f"{out_prefix}_confusion.png", dpi=150); plt.close(fig)


def _roc(scores, labels):
    order = np.argsort(-scores)
    y = labels[order]
    tp = np.cumsum(y); fp = np.cumsum(1 - y)
    tpr = tp / max(tp[-1], 1); fpr = fp / max(fp[-1], 1)
    return np.concatenate([[0], fpr]), np.concatenate([[0], tpr]), None


def _pr(scores, labels):
    order = np.argsort(-scores)
    y = labels[order]
    tp = np.cumsum(y); fp = np.cumsum(1 - y)
    prec = tp / np.maximum(tp + fp, 1); rec = tp / max(tp[-1], 1)
    return prec, rec, None


for tag, r in RESULTS.items():
    plot_report(r["res"], np.array(r["test_probs"]), np.array(r["test_labels"]), os.path.join(FIG_DIR, f"{tag}"))
print("plots saved to", FIG_DIR)


## 9. X-AI (Grad-CAM 3D/2D, occlusion, IG, attention, branch-drop)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

XAI_TAG = list(RESULTS.keys())[0]
_best = RESULTS[XAI_TAG]
_model = fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc2d_pretrained=False).to(DEVICE)
_model.load_state_dict(_best["model_state"])
_model.eval()

_va = FinalDS(DATA_ROOT, "Validation", (XAI_TAG.split("_s")[0] != "raw"), False)
_x, _v, _y = _va[0]
x3d = _x[None].to(DEVICE)
views = _v[None].to(DEVICE)
target = int(_y)

cam3d, cls, p = fm.grad_cam(_model.gcam3d_module(), _model, x3d, views, target=target, is_3d=True)
mid = cam3d.shape[0] // 2
vol = x3d[0, 0].detach().cpu().numpy()
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, fr in zip(axs, (0.4, 0.5, 0.6)):
    i = int(vol.shape[0] * fr)
    ax.imshow(vol[i], cmap="gray"); ax.imshow(cam3d[i], cmap="jet", alpha=0.45); ax.axis("off"); ax.set_title(f"z={i}")
fig.suptitle(f"Grad-CAM 3D ({XAI_TAG}, target={target}, p={p:.3f})")
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "xai_gradcam3d.png"), dpi=150); plt.close(fig)

for vi in range(N_2D):
    cam2d, _, _ = fm.grad_cam(_model.gcam2d_module(vi), _model, x3d, views, target=target, is_3d=False)
    img = views[0, vi, 0].detach().cpu().numpy()
    fig, axs = plt.subplots(1, 2, figsize=(8.5, 4))
    axs[0].imshow(img, cmap="gray"); axs[0].set_title("view"); axs[0].axis("off")
    axs[1].imshow(img, cmap="gray"); axs[1].imshow(cam2d, cmap="jet", alpha=0.45)
    axs[1].set_title(f"Grad-CAM 2D view{vi}"); axs[1].axis("off")
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, f"xai_gradcam2d_view{vi}.png"), dpi=150); plt.close(fig)

drop, base = fm.occlusion_sensitivity(_model, x3d, views, target, n=4)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(drop[:, :, drop.shape[2] // 2], cmap="hot"); ax.set_title("occlusion sensitivity (mid depth)")
ax.axis("off"); fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "xai_occlusion.png"), dpi=150); plt.close(fig)

attr = fm.integrated_gradients(_model, x3d, views, target, steps=16)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(attr[attr.shape[0] // 2], cmap="hot"); ax.set_title("integrated gradients (mid depth)")
ax.axis("off"); fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "xai_integrated_gradients.png"), dpi=150); plt.close(fig)

w, gate = fm.crossgate_attention(_model, x3d, views)
names, drops = fm.branch_drop_importance(_model, x3d, views, target)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
axs[0].bar([f"2D-{i}" for i in range(len(w))], w); axs[0].set_title(f"CrossGate attention (gate={gate:.3f})")
axs[1].bar(names, drops); axs[1].set_title("branch-drop importance")
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "xai_fusion_attention.png"), dpi=150); plt.close(fig)

with open(os.path.join(FIG_DIR, "xai_summary.json"), "w") as fh:
    json.dump({"tag": XAI_TAG, "target": target, "p": p, "gate": gate,
               "crossgate_weights": w.tolist(), "branch_drop": {n: d for n, d in zip(names, drops)}}, fh, indent=2)
print("X-AI figures saved to", FIG_DIR)


## 10. Lưu tất cả lên Drive + kết thúc

In [ ]:
def mount_drive():
    if os.path.isdir(DRIVE_ROOT):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return os.path.isdir(DRIVE_ROOT)
    except Exception as e:
        print("[drive] mount skipped:", e)
        return False


if mount_drive():
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.makedirs(DRIVE_FIG, exist_ok=True)
    for f in os.listdir(FIG_DIR):
        shutil.copy2(os.path.join(FIG_DIR, f), os.path.join(DRIVE_FIG, f))
    for tag, r in RESULTS.items():
        torch.save(r["model_state"], os.path.join(DRIVE_DIR, f"{tag}.pt"))
    print("[drive] figures ->", DRIVE_FIG)
    print("[drive] models  ->", DRIVE_DIR)
else:
    print("[drive] SKIP")
print("done")


## 11. Ghi chú & hạn chế

- **Khử nhiễu mặc định Bilateral** (`skimage.denoise_bilateral`, σ_color=0.10, σ_spatial=4.0, giữ biên tốt β≈0.80). Đổi `DENOISE_METHOD` sang `dncnn`/`swinir`/`tv`/`nlm` để thử.
- **Augmentation nhất quán**: cùng flip/rot90/jitter cho 3D và 2D (tránh lệch nhãn giữa nhánh).
- **Hậu xử lý**: threshold Youden-J trên val + temperature scaling; chọn checkpoint theo **val AUC**.
- **Chỉ số**: đầy đủ + bootstrap CI (1000 mẫu) cho AUC/PR/F1/balanced-acc/MCC.
- **Cần chạy ≥3 seed** (đã cấu hình) để báo cáo mean ± std; smoke chỉ 1 seed/1 epoch.
- So sánh **raw vs denoised** trên cùng split/seed; kết luận dựa trên AUC/PR-AUC/balanced-acc/MCC, không chỉ accuracy.
